# DTD 本地复现评测

基于 `DTD Reproduction on DocTamper.ipynb` 修改，从本地加载权重和数据集运行评测，无需 Google Colab / Drive。

**前置条件：**
- 权重文件：`pths/dtd_doctamper.pth`、`pths/vph_imagenet.pt`、`pths/swin_imagenet.pt`
- LMDB 数据集目录（如 `DocTamperV1-FCD`，包含 `data.mdb` 和 `lock.mdb`）
- CUDA + PyTorch 环境

In [ ]:
import os
import sys

PROJECT_ROOT = os.getcwd()
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')

# ==================== 用户配置区 ====================
# LMDB 数据集目录（包含 data.mdb 和 lock.mdb）
DATASET_DIR = os.path.join(PROJECT_ROOT, 'DocTamperV1-FCD')
LMDB_NAME = os.path.basename(DATASET_DIR)

# DTD 权重路径
PTH_PATH = os.path.join(PROJECT_ROOT, 'pths', 'dtd_doctamper.pth')

# 评测参数
MINQ = 75
BATCH_SIZE = 6
NUM_WORKERS = 0  # Windows 建议 0；Linux 可设 4~12
# ====================================================

for tag, p in [('models', MODELS_DIR), ('权重', PTH_PATH), ('数据集', DATASET_DIR)]:
    assert os.path.exists(p), f'{tag} 路径不存在: {p}'
    print(f'  {tag}: {p}')
sys.path.insert(0, MODELS_DIR)
print('\n环境检查通过')

: 

In [ ]:
# 如已安装可跳过本单元格
# 注：用 jpeglib 替代 jpegio（jpegio 在 Windows MSVC 下编译失败）
# segmentation_models_pytorch 0.2.x 需要 pretrainedmodels；用 --no-deps 时需单独装
!pip install lmdb albumentations tqdm opencv-python Pillow jpeglib
!pip install "pretrainedmodels==0.7.4"
!pip install --no-deps "segmentation_models_pytorch==0.2.1"
!pip install --no-deps "timm==0.4.12"
!pip install --no-deps "efficientnet_pytorch==0.7.1"

In [ ]:
import shutil
import subprocess

# 复制 qt_table.pk 到 models/ 目录
qt_src = os.path.join(PROJECT_ROOT, 'dtd', 'dtd', 'qt_table.pk')
qt_dst = os.path.join(MODELS_DIR, 'qt_table.pk')
if not os.path.exists(qt_dst):
    assert os.path.exists(qt_src), f'qt_table.pk 源文件不存在: {qt_src}'
    shutil.copy2(qt_src, qt_dst)
    print(f'已复制 qt_table.pk -> {qt_dst}')
else:
    print('qt_table.pk 已就绪')

# 生成 pks 文件（随机 JPEG 压缩链记录）
pks_dir = os.path.join(MODELS_DIR, 'pks')
os.makedirs(pks_dir, exist_ok=True)
pks_file = os.path.join(pks_dir, f'{LMDB_NAME}_{MINQ}.pk')
if not os.path.exists(pks_file):
    subprocess.run([
        sys.executable, os.path.join(PROJECT_ROOT, 'generate_pks.py'),
        '--lmdb_path', DATASET_DIR,
        '--minq', str(MINQ),
        '--out_dir', pks_dir,
    ], check=True)
    print(f'已生成: {pks_file}')
else:
    print(f'pks 已就绪: {pks_file}')

print('环境准备完成')

In [ ]:
import cv2
import lmdb
import torch
import jpeglib
import numpy as np
import pickle
import six
import tempfile
import types
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from albumentations.pytorch import ToTensorV2
import torchvision

# jpegio 在 Windows 上无法编译，注入基于 jpeglib 的兼容层
# 使得 dtd.py 中的 import jpegio 不会报错
if 'jpegio' not in sys.modules:
    _shim = types.ModuleType('jpegio')
    class _JpegioResult:
        def __init__(self, path):
            jpg = jpeglib.read_dct(path)
            b = jpg.Y
            self.coef_arrays = [b.transpose(0, 2, 1, 3).reshape(b.shape[0]*8, b.shape[1]*8)]
    _shim.read = lambda path: _JpegioResult(path)
    sys.modules['jpegio'] = _shim

from dtd import seg_dtd


class LocalTamperDataset(Dataset):
    """与 eval_dtd.py 中 TamperDataset 逻辑一致，但使用绝对路径以避免 CWD 依赖。"""

    def __init__(self, lmdb_path, lmdb_name, qt_path, pks_dir, minq=75):
        self.envs = lmdb.open(lmdb_path, max_readers=64, readonly=True,
                              lock=False, readahead=False, meminit=False)
        with self.envs.begin(write=False) as txn:
            self.nSamples = int(txn.get(b'num-samples'))
        self.max_nums = self.nSamples
        with open(qt_path, 'rb') as f:
            raw = pickle.load(f)
        self.pks = {k: torch.LongTensor(v) for k, v in raw.items()}
        with open(os.path.join(pks_dir, f'{lmdb_name}_{minq}.pk'), 'rb') as f:
            self.record = pickle.load(f)
        self.totsr = ToTensorV2()
        self.toctsr = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize((0.485, 0.455, 0.406), (0.229, 0.224, 0.225)),
        ])

    def __len__(self):
        return self.max_nums

    def __getitem__(self, index):
        with self.envs.begin(write=False) as txn:
            imgbuf = txn.get(f'image-{index:09d}'.encode())
            buf = six.BytesIO()
            buf.write(imgbuf)
            buf.seek(0)
            im = Image.open(buf)
            lblbuf = txn.get(f'label-{index:09d}'.encode())
            mask = (cv2.imdecode(np.frombuffer(lblbuf, dtype=np.uint8), 0) != 0).astype(np.uint8)
        record = self.record[index]
        choicei = len(record) - 1
        q = int(record[-1])
        use_qtb = self.pks[q]
        q_chain = []
        if choicei > 1:
            q_chain.append(int(record[-3]))
        if choicei > 0:
            q_chain.append(int(record[-2]))
        q_chain.append(q)
        mask = self.totsr(image=mask.copy())['image']
        # Windows 兼容的临时文件处理
        tmp_fd, tmp_path = tempfile.mkstemp(suffix='.jpg')
        os.close(tmp_fd)
        try:
            im = im.convert('L')
            for qv in q_chain:
                im.save(tmp_path, 'JPEG', quality=qv)
                im = Image.open(tmp_path).copy()
            import jpegio  # 使用上面注入的兼容层
            jpg = jpegio.read(tmp_path)
            dct = jpg.coef_arrays[0].copy()
        finally:
            try:
                os.unlink(tmp_path)
            except OSError:
                pass
        im = im.convert('RGB')
        return {
            'image': self.toctsr(im),
            'label': mask.long(),
            'rgb': np.clip(np.abs(dct), 0, 20),
            'q': use_qtb,
            'i': q,
        }


class IOUMetric:
    def __init__(self, n=2):
        self.n = n
        self.hist = np.zeros((n, n))

    def add_batch(self, preds, gts):
        for p, g in zip(preds, gts):
            f_g = g.flatten().astype(int)
            f_p = p.flatten()
            valid = (f_g >= 0) & (f_g < self.n)
            self.hist += np.bincount(
                self.n * f_g[valid] + f_p[valid], minlength=self.n ** 2
            ).reshape(self.n, self.n)

    def evaluate(self):
        return np.diag(self.hist) / (
            self.hist.sum(1) + self.hist.sum(0) - np.diag(self.hist)
        )


# 加载模型
print('正在加载模型...')
model = seg_dtd('', 2).cuda()
for m in model.modules():
    if isinstance(m, torch.nn.GELU) and not hasattr(m, 'approximate'):
        m.approximate = 'none'
model = torch.nn.DataParallel(model)
ckpt = torch.load(PTH_PATH, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print('模型加载完成')

In [ ]:
# 评测 DocTamperV1-FCD
qt_path = os.path.join(MODELS_DIR, 'qt_table.pk')
pks_dir = os.path.join(MODELS_DIR, 'pks')

test_data = LocalTamperDataset(DATASET_DIR, LMDB_NAME, qt_path, pks_dir, minq=MINQ)
loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)
print(f'数据集: {LMDB_NAME}  样本数: {len(test_data)}  minq={MINQ}')

iou = IOUMetric(2)
precs, recs = [], []

with torch.no_grad():
    for batch in tqdm(loader):
        data = batch['image'].cuda()
        target = batch['label'].cuda()
        dct = batch['rgb'].cuda()
        qs = batch['q'].unsqueeze(1).cuda()
        pred = model(data, dct, qs)
        predt = pred.argmax(1)
        targt = target.squeeze(1)
        matched = (predt * targt).sum((1, 2))
        precs.append((matched / (predt.sum((1, 2)) + 1e-8)).mean().item())
        recs.append((matched / targt.sum((1, 2))).mean().item())
        iou.add_batch(pred.cpu().numpy().argmax(1), target.cpu().numpy())

iu = iou.evaluate()
pre = np.mean(precs)
rec = np.mean(recs)
f1 = 2 * pre * rec / (pre + rec + 1e-8)
print(f'[val] iou:{iu} pre:{pre} rec:{rec} f1:{f1}')

In [ ]:
# （可选）评测 DocTamperV1-SCD
import subprocess

SCD_DIR = os.path.join(PROJECT_ROOT, 'DocTamperV1-SCD')
SCD_NAME = os.path.basename(SCD_DIR)
assert os.path.exists(SCD_DIR), f'SCD 数据集不存在: {SCD_DIR}'

scd_pks = os.path.join(pks_dir, f'{SCD_NAME}_{MINQ}.pk')
if not os.path.exists(scd_pks):
    subprocess.run([
        sys.executable, os.path.join(PROJECT_ROOT, 'generate_pks.py'),
        '--lmdb_path', SCD_DIR,
        '--minq', str(MINQ),
        '--out_dir', pks_dir,
    ], check=True)

scd_data = LocalTamperDataset(SCD_DIR, SCD_NAME, qt_path, pks_dir, minq=MINQ)
scd_loader = DataLoader(scd_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)
print(f'数据集: {SCD_NAME}  样本数: {len(scd_data)}  minq={MINQ}')

iou_scd = IOUMetric(2)
precs_s, recs_s = [], []

with torch.no_grad():
    for batch in tqdm(scd_loader):
        data = batch['image'].cuda()
        target = batch['label'].cuda()
        dct = batch['rgb'].cuda()
        qs = batch['q'].unsqueeze(1).cuda()
        pred = model(data, dct, qs)
        predt = pred.argmax(1)
        targt = target.squeeze(1)
        matched = (predt * targt).sum((1, 2))
        precs_s.append((matched / (predt.sum((1, 2)) + 1e-8)).mean().item())
        recs_s.append((matched / targt.sum((1, 2))).mean().item())
        iou_scd.add_batch(pred.cpu().numpy().argmax(1), target.cpu().numpy())

iu_s = iou_scd.evaluate()
pre_s = np.mean(precs_s)
rec_s = np.mean(recs_s)
f1_s = 2 * pre_s * rec_s / (pre_s + rec_s + 1e-8)
print(f'[val] iou:{iu_s} pre:{pre_s} rec:{rec_s} f1:{f1_s}')